# Mini Project 1 — CourtListener Citation Analysis

**Name:** Ranjitha Rangaswamy  
**Dataset:** `data/courtlistener.csv` (CourtListener API export)  
**Date:** May 2026

This folder is a **standalone** mini project: data, charts, and notebook live in `MiniProject1/` so reviewers can open one directory on GitHub without hunting other weeks.


In [1]:
# Setup — run this cell first (required)
!pip install jupyter plotly kaleido pandas

import subprocess
import sys
from pathlib import Path

# Fallback if shell `pip` is not on PATH (common in some Jupyter shells)
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "jupyter", "plotly", "kaleido", "pandas", "-q"],
    stdout=subprocess.DEVNULL,
)

import pandas as pd

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "data" / "courtlistener.csv").exists():
    alt = PROJECT_DIR / "MiniProject1"
    if (alt / "data" / "courtlistener.csv").exists():
        PROJECT_DIR = alt
        sys.path.insert(0, str(PROJECT_DIR))

sys.path.insert(0, str(PROJECT_DIR))
import mp1_charts as charts

DATA_CSV = PROJECT_DIR / "data" / "courtlistener.csv"
IMAGES_DIR = PROJECT_DIR / "images"
print("Project dir:", PROJECT_DIR.resolve())
print("Data file exists:", DATA_CSV.exists())


Project dir: /Users/rnr6/Documents/HCDE/hcde530/MiniProject1
Data file exists: True


---

## Section 1 — Overview

### What is this dataset?

This analysis uses **CourtListener** — a free legal research platform — exported to `data/courtlistener.csv` (~**39,000 rows**). Each row is one opinion with `case` (title), `judge`, `court` / `court_id`, `date_of_decision`, and `cite_count` (incoming citation clusters in this extract).

The file combines two Seattle-area searches:

- **Federal:** U.S. District Court, Western District of Washington (`wawd`) — ~3,000 rows; **non-zero** `cite_count` in this pull.
- **State:** Washington Court of Appeals (`washctapp`) — ~36,000 rows; here **`cite_count` is 0 for every row** and `judge` is mostly missing.

**Source:** [CourtListener API](https://www.courtlistener.com/help/api/) search/export, stacked in earlier course weeks and copied into this folder for MP1.

### Why these questions matter

I want to see **which opinions carry citation signal in public data** and whether that differs by court level — relevant for legal reference workflows and for HCD practice (always label which population a metric describes).

### Three analytical questions (MP1a)

1. Which judges author the most cited opinions — and does citation frequency vary by **court level**? *(Shown as mean cites by month and court; judges are only reliable on the federal slice.)*
2. What are the **most cited judgments** in a specific jurisdiction? *(Western District of Washington.)*
3. What is the **average** cite behavior per opinion row, and which **case titles** accumulate the most citation mass?

**Practical use:** A lawyer can scan dominant cited titles in this territory; the analysis also warns when state and federal rows **cannot** be compared on `cite_count` without relabeling.

**Week 6 vs this folder:** `week 6/Week6_files/week6_mini_analytics.ipynb` only asks **mini** questions (e.g. how many rows fall in 2025, federal vs state counts). Those three bullets above are **only** developed here in Mini Project 1.


---

## Section 2 — Data Profile

Run four first-look operations. One sentence interprets each (assignment requirement).


In [2]:
df = charts.load_df()
print(f"Loaded {len(df):,} rows from {DATA_CSV.name}")
df.head()


Loaded 39,040 rows from courtlistener.csv


,dataset,case,judge,court,court_id,date_of_decision,cite_count,court_level,judge_clean
0,King County (Wash. Ct. App.),"State Of Washington, V. Justin R. Smith",NaN,Court of Appeals of Washington,washctapp,2025-07-21,0,State: Wash. Court of Appeals (King County pull),NaN
1,King County (Wash. Ct. App.),"Chen Wang And Liyin Xue, V. Dongmei Huang",NaN,Court of Appeals of Washington,washctapp,2025-08-25,0,State: Wash. Court of Appeals (King County pull),NaN
2,King County (Wash. Ct. App.),In the Matter of the Detention of C.E.,NaN,Court of Appeals of Washington,washctapp,2025-09-09,0,State: Wash. Court of Appeals (King County pull),NaN
3,King County (Wash. Ct. App.),"Erwin Chappel, Respondent/cr-appellants V. Dou...",NaN,Court of Appeals of Washington,washctapp,2025-09-15,0,State: Wash. Court of Appeals (King County pull),NaN
4,King County (Wash. Ct. App.),"Alaska Airlines, V. Hillary Spanjer",NaN,Court of Appeals of Washington,washctapp,2025-09-15,0,State: Wash. Court of Appeals (King County pull),NaN


In [3]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 39040 entries, 0 to 39039
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   dataset           39040 non-null  str           
 1   case              39040 non-null  str           
 2   judge             2888 non-null   str           
 3   court             39040 non-null  str           
 4   court_id          39040 non-null  str           
 5   date_of_decision  39040 non-null  datetime64[us]
 6   cite_count        39040 non-null  int64         
 7   court_level       39040 non-null  str           
 8   judge_clean       2888 non-null   str           
dtypes: datetime64[us](1), int64(1), str(7)
memory usage: 2.7 MB


In [4]:
df.describe(include="all")


,dataset,case,judge,court,court_id,date_of_decision,cite_count,court_level,judge_clean
count,39040,39040,2888,39040,39040,39040,39040.000000,39040,2888
unique,2,40,10,2,2,NaN,NaN,2,10
top,King County (Wash. Ct. App.),"State Of Washington, V. Justin R. Smith",Settle,Court of Appeals of Washington,washctapp,NaN,NaN,State: Wash. Court of Appeals (King County pull),Settle
freq,36000,1800,608,36000,36000,NaN,NaN,36000,608
mean,NaN,NaN,NaN,NaN,NaN,2025-02-05 20:57:56.065573,0.151844,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,2018-10-31 00:00:00,0.000000,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,2025-06-26 00:00:00,0.000000,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,2025-08-05 00:00:00,0.000000,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,2025-09-02 00:00:00,0.000000,NaN,NaN
max,NaN,NaN,NaN,NaN,NaN,2025-09-22 00:00:00,7.000000,NaN,NaN


In [5]:
missing = df.isnull().sum()
print(missing)
print("judge_clean missing:", df["judge_clean"].isna().sum())


dataset                 0
case                    0
judge               36152
court                   0
court_id                0
date_of_decision        0
cite_count              0
court_level             0
judge_clean         36152
dtype: int64
judge_clean missing: 36152


### Data profile interpretation

| Operation | One-sentence interpretation |
|-----------|----------------------------|
| **`head()`** | Each row is one opinion with a long case title, court metadata, a decision date, and a numeric cite count. |
| **`info()`** | Seven columns and ~39k rows; dates parse cleanly; judge is null on most rows because the state bulk dominates the file. |
| **`describe()`** | Overall mean cite is ~0.15 because ~36k state rows are zero; federal rows average ~1.95 cites in this extract. |
| **`isnull().sum()`** | Missing judges and all-zero state cites are **coverage** issues — they shape which questions we can answer without misleading aggregates.


---

## Section 3 — Analysis

Three charts answer the MP1a questions. Each has a Plotly figure in-notebook, a **static JPG** in `images/` (Kaleido), and an interpretation of what the chart **argues**.


### Question 1 — Citations by court level over time

Does citation frequency vary by court level? (Judge-level view is limited to the federal slice in this CSV.)


In [6]:
fig_q1 = charts.fig_q1_scatter3d_monthly(df)
fig_q1.show(config=charts.GRAPH_CONFIG)


**What this chart argues:** The state appellate slice stays at **mean cite = 0** over time; the federal district shows **non-zero** monthly means. Court level must be named before anyone infers how “cited” local opinions are.


### Question 2 — Most cited judgments (W.D. Wash.)

Top case titles in the federal district by max `cite_count` in this pull.


In [7]:
fig_q2 = charts.fig_q2_scatter3d_jurisdiction(df, court_id="wawd", top_n=45, rank_mode="max")
fig_q2.show(config=charts.GRAPH_CONFIG)


**What this chart argues:** Citation strength in `wawd` is **concentrated at the top ranks**; many rows repeat the same case caption (depth on the z-axis). Useful as a **starting list**, not a full citator replacement.


### Question 3 — Top authority titles (dataset-wide)

Summed `cite_count` by case title; title states the mean cite per row.


In [8]:
fig_q3 = charts.fig_q3_top_authorities(df, top_n=45, min_opinion_rows=1)
fig_q3.show(config=charts.GRAPH_CONFIG)


**What this chart argues:** The **global mean** is low because of zero-heavy state rows, but **total cite mass** still piles onto a federal-heavy long tail of titles — “average row” and “top authority” are different questions.


In [9]:
# Save static chart images to images/ (also committed for reviewers)
try:
    written = charts.export_chart_images(fig_q1, fig_q2, fig_q3, out_dir=IMAGES_DIR)
    for p in written:
        print(p.relative_to(PROJECT_DIR))
except ImportError as e:
    print("Kaleido export skipped:", e)
    print("Using committed JPGs in images/")
    for p in sorted(IMAGES_DIR.glob("mp1a_chart*.jpg")):
        print(p.relative_to(PROJECT_DIR))


wrote /Users/rnr6/Documents/HCDE/hcde530/MiniProject1/images/mp1a_chart1_court_level_cites_3d.jpg


wrote /Users/rnr6/Documents/HCDE/hcde530/MiniProject1/images/mp1a_chart2_top_cited_cases_wd_wash.jpg


wrote /Users/rnr6/Documents/HCDE/hcde530/MiniProject1/images/mp1a_chart3_top_authority_titles.jpg
images/mp1a_chart1_court_level_cites_3d.jpg
images/mp1a_chart2_top_cited_cases_wd_wash.jpg
images/mp1a_chart3_top_authority_titles.jpg


### Static chart files (also in `images/`)

![Chart 1 — court level over time](images/mp1a_chart1_court_level_cites_3d.jpg)

![Chart 2 — top cited cases W.D. Wash.](images/mp1a_chart2_top_cited_cases_wd_wash.jpg)

![Chart 3 — top authority titles](images/mp1a_chart3_top_authority_titles.jpg)


---

## Section 4 — Conclusions

### Question 1

Federal rows carry the only meaningful `cite_count` variation; state rows read as uncited **in this extract**. I would not claim Washington appellate opinions are never cited in reality — only that this field is empty here. Next: confirm another API field for state cites or restrict dashboards to `wawd`.

### Question 2

Within W.D. Wash., a **steep head** of titles shows the highest max cites, often with duplicate rows per caption. Use as a bibliography starter, then verify in a dedicated citator.

### Question 3

The **global mean** (~0.15) is dragged down by `washctapp` zeros; **summed bars** show which titles still accumulate cite mass (mostly federal-driven). Further work: per-court means, deduplicate captions, document CourtListener’s cite cluster definition.

### Further investigation

- Populate or explain missing state cite metadata.
- Federal-only judge ranking with explicit population notes.
- Data card table (rows, % missing judge, mean cite by court) above every chart.


---

## Section 5 — Process

**Standalone folder:** I copied `courtlistener.csv` and chart logic into `MiniProject1/` so graders need only this directory.

**Weeks 4–6:** API stacking (Week 5), then Plotly/Dash (Week 6). The important pivot was discovering **36k `washctapp` rows with cite_count = 0** — I stopped implying one aggregate speaks for all courts.

**Tooling:** Cursor helped with Dash layout; invalid Mantine props still failed at runtime until fixed. Kaleido exports required a venv/`pip install kaleido`; this notebook’s setup cell installs dependencies for a clean **Restart & Run All**.

**AI:** Suggested extra judge charts; I dropped them so each figure maps to one MP1a question. Conclusions here reflect the corrected story after the slice-quality check.
